In [16]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [41]:
import pandas as pd
import os

def run_full_pipeline(excel_file, target_images=250):

    food_list = load_and_clean_food_items(excel_file)

    processed_dir = "processed_images"
    os.makedirs(processed_dir, exist_ok=True)

    existing_images = [
        f for f in os.listdir(processed_dir)
        if f.lower().endswith(".jpg")
    ]

    processed_count = len(existing_images)

    print(f"Already processed: {processed_count}")
    print(f"Target: {target_images}")
    print(f"Remaining: {target_images - processed_count}")

    report = []

    for index, food in enumerate(food_list, start=1):

        if processed_count >= target_images:
            print("\nTARGET REACHED!")
            print(f"Total processed images: {processed_count}")
            break

        original_name = food["original_name"]
        search_query = food["search_query"]

        clean_name = sanitize_filename(original_name)

        processed_path = os.path.join(
            processed_dir,
            f"{clean_name}.jpg"
        )

        if os.path.exists(processed_path):

            print(f"\n[{index}] SKIPPED:")
            print(original_name)

            continue

        print("\n" + "=" * 60)
        print(f"PROCESSING")
        print(f"Food: {original_name}")
        print(f"Progress: {processed_count}/{target_images}")
        print("=" * 60)

        raw_path = search_and_download_image(
            search_query
        )

        if raw_path:

            processed_path = process_and_resize_image(
                raw_path,
                original_name
            )

            if processed_path:
                processed_count += 1

                print(
                    f"SUCCESS: {processed_count}/{target_images}"
                )

                report.append({
                    "food_name": original_name,
                    "search_query": search_query,
                    "raw_image": raw_path,
                    "processed_image": processed_path,
                    "status": "SUCCESS"
                })

            else:

                report.append({
                    "food_name": original_name,
                    "search_query": search_query,
                    "raw_image": raw_path,
                    "processed_image": None,
                    "status": "PROCESSING_FAILED"
                })

        else:

            report.append({
                "food_name": original_name,
                "search_query": search_query,
                "raw_image": None,
                "processed_image": None,
                "status": "DOWNLOAD_FAILED"
            })

    if report:

        report_df = pd.DataFrame(report)

        os.makedirs("reports", exist_ok=True)

        report_path = os.path.join(
            "reports",
            "processing_report.xlsx"
        )

        report_df.to_excel(
            report_path,
            index=False
        )

        print(f"\nReport saved to: {report_path}")

    print("\n" + "=" * 60)
    print("PIPELINE STOPPED")
    print("=" * 60)

    print(f"Processed images: {processed_count}")
    print(f"Target: {target_images}")

In [1]:
pip install duckduckgo_search requests

   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.4 MB 3.5 MB/s eta 0:00:02
   ---- ----------------------------------- 0.8/6.4 MB 2.0 MB/s eta 0:00:03
   ------ --------------------------------- 1.0/6.4 MB 1.7 MB/s eta 0:00:04
   -------- ------------------------------- 1.3/6.4 MB 1.6 MB/s eta 0:00:04
   ----------- ---------------------------- 1.8/6.4 MB 1.6 MB/s eta 0:00:03
   ------------- -------------------------- 2.1/6.4 MB 1.7 MB/s eta 0:00:03
   ---------------- ----------------------- 2.6/6.4 MB 1.8 MB/s eta 0:00:03
   ------------------- -------------------- 3.1/6.4 MB 1.9 MB/s eta 0:00:02
   ----------------------- ---------------- 3.7/6.4 MB 2.0 MB/s eta 0:00:02
   -------------------------- ------------- 4.2/6.4 MB 2.1 MB/s eta 0:00:02
   ------------------------------- -------- 5.0/6.4 MB 2.2 MB/s eta 0:00:01
   ------------------------------------- -- 6.0/6.4 MB 2.4 MB/s eta 0:00:01
   ----------------

In [1]:
!pip uninstall -y duckduckgo_search
!pip install -U ddgs

In [18]:
from ddgs import DDGS

print("DDGS imported successfully")

DDGS imported successfully


In [19]:
from ddgs import DDGS

query = "Chicken Tandoori dish restaurant high quality photo"

with DDGS() as ddgs:
    results = list(ddgs.images(query, max_results=3))

print("Number of results:", len(results))

for result in results:
    print(result.get("image"))

Number of results: 3
https://static.vecteezy.com/system/resources/previews/029/894/950/large_2x/of-tandoori-chicken-as-a-dish-in-a-high-end-restaurant-generative-ai-photo.jpg
https://static.vecteezy.com/system/resources/previews/029/287/405/non_2x/of-tandoori-chicken-as-a-dish-in-a-high-end-restaurant-generative-ai-photo.jpg
https://static.vecteezy.com/system/resources/previews/029/287/410/large_2x/of-tandoori-chicken-as-a-dish-in-a-high-end-restaurant-generative-ai-photo.jpg


In [20]:
import os
import re
import requests
from ddgs import DDGS


def sanitize_filename(filename):
    """Remove characters that are invalid in Windows filenames."""
    return re.sub(r'[\\/*?:"<>|]', "", filename).strip()


def search_and_download_image(
    food_name,
    output_dir="raw_images",
    blur_threshold=80.0
):
    """
    Searches for multiple images, downloads candidates,
    and keeps the first image that passes the sharpness test.
    """

    os.makedirs(output_dir, exist_ok=True)

    clean_name = sanitize_filename(food_name)

    save_path = os.path.join(
        output_dir,
        f"{clean_name}_raw.jpg"
    )

    query = f"{food_name} dish restaurant high quality photo"

    print(f"\nSearching image for: {food_name}...")

    try:

        # Search for 3 image candidates
        with DDGS() as ddgs:
            results = list(
                ddgs.images(
                    query,
                    max_results=3
                )
            )

        if not results:
            print(f"[FAILED] No images found for: {food_name}")
            return None

        # Try each candidate
        for idx, result in enumerate(results, start=1):

            image_url = result.get("image")

            if not image_url:
                continue

            print(f"Trying image {idx}...")

            try:

                # Download candidate
                response = requests.get(
                    image_url,
                    timeout=7,
                    headers={
                        "User-Agent": "Mozilla/5.0"
                    }
                )

                if response.status_code != 200:
                    print(
                        f"[SKIPPED] Image {idx}: "
                        f"HTTP {response.status_code}"
                    )
                    continue

                # Check that the response is actually an image
                content_type = response.headers.get(
                    "Content-Type",
                    ""
                )

                if not content_type.startswith("image/"):
                    print(
                        f"[SKIPPED] Image {idx}: "
                        "not a valid image"
                    )
                    continue

                with open(save_path, "wb") as f:
                    f.write(response.content)

                if not is_image_sharp(
                    save_path,
                    blur_threshold
                ):
                    print(
                        f"[REJECTED] Image {idx} "
                        "is too blurry."
                    )

                    os.remove(save_path)

                    continue

                print(
                    f"[ACCEPTED] Image {idx} "
                    "passed quality check."
                )

                return save_path

            except Exception as e:

                print(
                    f"[SKIPPED] Image {idx} failed: {e}"
                )

                continue

    except Exception as e:

        print(
            f"[ERROR] Search failed for "
            f"{food_name}: {e}"
        )

    print(
        f"[FAILED] No suitable image found "
        f"for {food_name}"
    )

    return None

In [4]:
search_and_download_image("Chicken Tandoori")


Searching image for: Chicken Tandoori...
Trying image 1...
Blur score (higher is sharper): 51.02
[REJECTED] Image 1 is too blurry.
Trying image 2...
Blur score (higher is sharper): 91.38
[ACCEPTED] Image 2 passed quality check.


'raw_images\\Chicken Tandoori_raw.jpg'

In [5]:
!pip install -U pillow opencv-python

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.2 MB 2.5 MB/s eta 0:00:03
   ------- -------------------------------- 1.3/7.2 MB 2.5 MB/s eta 0:00:03
   ---------- ----------------------------- 1.8/7.2 MB 2.6 MB/s eta 0:00:03
   -------------- ------------------------- 2.6/7.2 MB 2.6 MB/s eta 0:00:02
   ----------------- ---------------------- 3.1/7.2 MB 2.8 MB/s eta 0:00:02
   --------------------- ------------------ 3.9/7.2 MB 2.8 MB/s eta 0:00:02
   -------------------------- ------------- 4.7/7.2 MB 2.9 MB/s eta 0:00:01
   ---------------------------- ----------- 5.2/7.2 MB 3.0 MB/s eta 0:00:01
   ------------------------------- -------- 5.8/7.2 MB 3.0 MB/s eta 0:00:01
   ------------------------------------ --- 6.6/7.2 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 3.0 MB/s eta 0:00:00
   -----------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pillow<12,>=7.1.0, but you have pillow 12.3.0 which is incompatible.


In [21]:
from PIL import Image
import cv2

print("Image processing tools are working!")

Image processing tools are working!


In [22]:
import os
import cv2
from PIL import Image, ImageOps

def is_image_sharp(image_path, blur_threshold=80.0):
    """Calculates Laplacian variance to detect if an image is too blurry."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False
    variance = cv2.Laplacian(img, cv2.CV_64F).var()
    print(f"Blur score (higher is sharper): {variance:.2f}")
    return variance >= blur_threshold

import os
from PIL import Image, ImageOps


def process_and_resize_image(
    input_path,
    food_name,
    output_dir="processed_images",
    target_size=(1800, 1200)
):
    """
    Resizes, center-crops, and compresses an image
    to 1800x1200 and keeps it under 10 MB.
    """

    os.makedirs(output_dir, exist_ok=True)

    clean_name = sanitize_filename(food_name)

    save_path = os.path.join(
        output_dir,
        f"{clean_name}.jpg"
    )

    try:

        with Image.open(input_path) as img:

            print(f"Original size: {img.size}")

            # Convert image to RGB
            if img.mode != "RGB":
                img = img.convert("RGB")

            # Resize + center crop
            processed_img = ImageOps.fit(
                img,
                target_size,
                Image.Resampling.LANCZOS,
                centering=(0.5, 0.5)
            )

            print(
                f"Processed size: {processed_img.size}"
            )

            # Start with high JPEG quality
            quality = 90

            processed_img.save(
                save_path,
                "JPEG",
                quality=quality,
                optimize=True
            )

            while (
                os.path.getsize(save_path)
                > 10 * 1024 * 1024
                and quality > 10
            ):

                quality -= 5

                processed_img.save(
                    save_path,
                    "JPEG",
                    quality=quality,
                    optimize=True
                )

            file_size_mb = (
                os.path.getsize(save_path)
                / (1024 * 1024)
            )

            print(
                f"[PROCESSED & SAVED] {save_path}"
            )

            print(
                f"Final size: "
                f"{processed_img.size[0]} × "
                f"{processed_img.size[1]} pixels"
            )

            print(
                f"File size: {file_size_mb:.2f} MB"
            )

            return save_path

    except Exception as e:

        print(
            f"[ERROR] Failed to process "
            f"'{food_name}': {e}"
        )

        return None

if __name__ == "__main__":
    raw_path = "raw_images/Chicken Tandoori_raw.jpg"
    if os.path.exists(raw_path):
        process_and_resize_image(raw_path, "Chicken Tandoori")

Original size: (3497, 1960)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Chicken Tandoori.jpg
Final size: 1800 × 1200 pixels
File size: 0.39 MB


In [23]:
process_and_resize_image(
    raw_path,
    "Chicken Tandoori"
)

Original size: (3497, 1960)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Chicken Tandoori.jpg
Final size: 1800 × 1200 pixels
File size: 0.39 MB


'processed_images\\Chicken Tandoori.jpg'

In [9]:
!pip install -U pandas openpyxl

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.8 MB 1.3 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/9.8 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.8 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.8 MB 1.3 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/9.8 MB 1.3 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.8 MB 1.3 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/9.8 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.4/9.8 MB 1.3 MB/s eta 0:00:06
   ----------- ---------------------------- 2.9/9.8 MB 1.4 MB/s eta 0:00:05
   ------------ --------------------------- 3.1/9.8 MB 1.4 MB/s eta 0:00:05
   -------------- ------------------------- 3.7/9.8 MB 1.5 MB/s eta 0:00:05
   ---------------- ------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.45.1 requires pillow<12,>=7.1.0, but you have pillow 12.3.0 which is incompatible.


In [10]:
!pip install -U google-api-python-client google-auth-httplib2 google-auth-oauthlib

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
    --------------------------------------- 0.3/16.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.1 MB 1.3 MB/s eta 0:00:13
   - -------------------------------------- 0.5/16.1 MB 1.3 MB/s eta 0:00:13
   - -------------------------------------- 0.8/16.1 MB 844.6 kB/s eta 0:00:19
   -- ------------------------------------- 1.0/16.1 MB 1.1 MB/s eta 0:00:15
   --- ------------------------------------ 1.3/16.1 MB 1.1 MB/s eta 0:00:14
   --- ------------------------------------ 1.6/16.1 MB 1.1 MB/s eta 0:00:14
   ---- ----------------------------------- 1.8/16.1 MB 1.1 MB/s eta 0:00:13
   ----- ---------------------------------- 2.1/16.1 MB 1.2 MB/s eta 0:00:12
   ----- ---------------------------------- 2.4/16.1 MB 1.2 MB/s eta 0:00:12
   ------ --------------------------------- 2.6/16.1 MB 1.2 MB/s eta 0:00:12
   ------- -------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.45.1 requires pillow<12,>=7.1.0, but you have pillow 12.3.0 which is incompatible.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.36.1 which is incompatible.


In [24]:
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials

print("Google Drive libraries are working!")

Google Drive libraries are working!


In [25]:
import pandas as pd
import os


def run_food_image_pipeline(excel_file):

    # -----------------------------------------
    # STEP 1: Load food items from Excel
    # -----------------------------------------

    food_list = load_and_clean_food_items(excel_file)

    print(f"\nTotal food items found: {len(food_list)}")

    report = []

    # -----------------------------------------
    # STEP 2: Process each food item
    # -----------------------------------------

    for index, food in enumerate(food_list, start=1):

        original_name = food["original_name"]
        search_query = food["search_query"]

        print("\n" + "=" * 60)
        print(f"PROCESSING {index}/{len(food_list)}")
        print(f"Food: {original_name}")
        print("=" * 60)

        raw_path = None
        processed_path = None
        status = "FAILED"

        # -----------------------------------------
        # STEP 3: Search and download
        # -----------------------------------------

        raw_path = search_and_download_image(
            search_query
        )

        if raw_path:

            # -----------------------------------------
            # STEP 4: Process image
            # -----------------------------------------

            processed_path = process_and_resize_image(
                raw_path,
                original_name
            )

            if processed_path:
                status = "SUCCESS"

        # -----------------------------------------
        # STEP 5: Add result to report
        # -----------------------------------------

        report.append({
            "food_name": original_name,
            "search_query": search_query,
            "raw_image": raw_path,
            "processed_image": processed_path,
            "status": status
        })

    # -----------------------------------------
    # STEP 6: Create report
    # -----------------------------------------

    report_df = pd.DataFrame(report)

    os.makedirs("reports", exist_ok=True)

    report_path = os.path.join(
        "reports",
        "processing_report.xlsx"
    )

    report_df.to_excel(
        report_path,
        index=False
    )

    # -----------------------------------------
    # STEP 7: Summary
    # -----------------------------------------

    successful = sum(
        report_df["status"] == "SUCCESS"
    )

    failed = sum(
        report_df["status"] == "FAILED"
    )

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print("=" * 60)

    print(f"Total items : {len(report_df)}")
    print(f"Successful  : {successful}")
    print(f"Failed      : {failed}")

    print(f"\nReport saved to:")
    print(report_path)

    return report_df

In [34]:
excel_file = r"C:\Users\Bharg\Downloads\Assignment - Ai agent  - Sheet1.xlsx"

In [29]:
food_list = load_and_clean_food_items(excel_file)

print("First 3 items:")
print(food_list[:3])

First 3 items:
[{'original_name': 'Chicken Tandoori', 'search_query': 'Chicken Tandoori'}, {'original_name': 'Veg Fried Rice', 'search_query': 'Veg Fried Rice'}, {'original_name': 'Chicken Biryani Boneless', 'search_query': 'Chicken Biryani Boneless'}]


In [30]:
def process_test_food_items(excel_file, number_of_items=3):

    food_list = load_and_clean_food_items(excel_file)

    test_list = food_list[:number_of_items]

    print(f"\nTesting {len(test_list)} food items...")

    for index, food in enumerate(test_list, start=1):

        original_name = food["original_name"]
        search_query = food["search_query"]

        print("\n" + "=" * 60)
        print(f"PROCESSING {index}/{len(test_list)}")
        print(f"Food: {original_name}")
        print("=" * 60)

        # --------------------------------
        # SEARCH + DOWNLOAD
        # --------------------------------

        raw_path = search_and_download_image(
            search_query
        )

        # --------------------------------
        # RESIZE + CROP + COMPRESS
        # --------------------------------

        if raw_path:

            processed_path = process_and_resize_image(
                raw_path,
                original_name
            )

            if processed_path:

                print(
                    f"[COMPLETE] {original_name}"
                )

            else:

                print(
                    f"[FAILED] Processing: "
                    f"{original_name}"
                )

        else:

            print(
                f"[FAILED] Downloading: "
                f"{original_name}"
            )

In [32]:
process_test_food_items(
    excel_file,
    number_of_items=3
)


Testing 3 food items...

PROCESSING 1/3
Food: Chicken Tandoori

Searching image for: Chicken Tandoori...
Trying image 1...
Blur score (higher is sharper): 51.02
[REJECTED] Image 1 is too blurry.
Trying image 2...
Blur score (higher is sharper): 91.38
[ACCEPTED] Image 2 passed quality check.
Original size: (3497, 1960)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Chicken Tandoori.jpg
Final size: 1800 × 1200 pixels
File size: 0.39 MB
[COMPLETE] Chicken Tandoori

PROCESSING 2/3
Food: Veg Fried Rice

Searching image for: Veg Fried Rice...
Trying image 1...
Blur score (higher is sharper): 126.79
[ACCEPTED] Image 1 passed quality check.
Original size: (1050, 700)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Veg Fried Rice.jpg
Final size: 1800 × 1200 pixels
File size: 0.27 MB
[COMPLETE] Veg Fried Rice

PROCESSING 3/3
Food: Chicken Biryani Boneless

Searching image for: Chicken Biryani Boneless...
Trying image 1...
Blur score (higher is sharper): 372.

In [36]:
def run_full_pipeline(excel_file):

    food_list = load_and_clean_food_items(excel_file)

    print(f"\nTotal food items: {len(food_list)}")

    report = []

    for index, food in enumerate(food_list, start=1):

        original_name = food["original_name"]
        search_query = food["search_query"]

        print("\n" + "=" * 60)
        print(f"PROCESSING {index}/{len(food_list)}")
        print(f"Food: {original_name}")
        print("=" * 60)

        raw_path = None
        processed_path = None
        status = "FAILED"

        raw_path = search_and_download_image(
            search_query
        )

        if raw_path:

            processed_path = process_and_resize_image(
                raw_path,
                original_name
            )

            if processed_path:
                status = "SUCCESS"

        report.append({
            "food_name": original_name,
            "search_query": search_query,
            "raw_image": raw_path,
            "processed_image": processed_path,
            "status": status
        })

    report_df = pd.DataFrame(report)

    os.makedirs("reports", exist_ok=True)

    report_path = os.path.join(
        "reports",
        "processing_report.xlsx"
    )

    report_df.to_excel(
        report_path,
        index=False
    )

    successful = (
        report_df["status"] == "SUCCESS"
    ).sum()

    failed = (
        report_df["status"] == "FAILED"
    ).sum()

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print("=" * 60)

    print(f"Total     : {len(report_df)}")
    print(f"Successful: {successful}")
    print(f"Failed    : {failed}")

    print(f"\nReport:")
    print(report_path)

    return report_df

In [35]:
report_df = run_full_pipeline(excel_file)


Total food items: 420

PROCESSING 1/420
Food: Chicken Tandoori

Searching image for: Chicken Tandoori...
Trying image 1...
Blur score (higher is sharper): 51.02
[REJECTED] Image 1 is too blurry.
Trying image 2...
Blur score (higher is sharper): 526.42
[ACCEPTED] Image 2 passed quality check.
Original size: (1749, 980)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Chicken Tandoori.jpg
Final size: 1800 × 1200 pixels
File size: 0.37 MB

PROCESSING 2/420
Food: Veg Fried Rice

Searching image for: Veg Fried Rice...
Trying image 1...
Blur score (higher is sharper): 126.79
[ACCEPTED] Image 1 passed quality check.
Original size: (1050, 700)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Veg Fried Rice.jpg
Final size: 1800 × 1200 pixels
File size: 0.27 MB

PROCESSING 3/420
Food: Chicken Biryani Boneless

Searching image for: Chicken Biryani Boneless...
Trying image 1...
Blur score (higher is sharper): 372.77
[ACCEPTED] Image 1 passed quality check.
Origin

KeyboardInterrupt: 

In [38]:
excel_file = r"C:\Users\Bharg\Downloads\Assignment - Ai agent  - Sheet1.xlsx"

report_df = run_full_pipeline(
    excel_file,
    max_items=250
)


Total food items in Excel: 420

[1] SKIPPED - Already processed:
Chicken Tandoori

[2] SKIPPED - Already processed:
Veg Fried Rice

[3] SKIPPED - Already processed:
Chicken Biryani Boneless

[4] SKIPPED - Already processed:
Butter Roti

[5] SKIPPED - Already processed:
Paneer Tikka Masala

[6] SKIPPED - Already processed:
Chicken Fried Rice

[7] SKIPPED - Already processed:
Veg Triple Schezwan Fried Rice

[8] SKIPPED - Already processed:
Roti

[9] SKIPPED - Already processed:
Chicken Tandoori

[10] SKIPPED - Already processed:
Dal Khichdi Tadka Chicken Triple Fried Rice

[11] SKIPPED - Already processed:
Chicken Lollipop

[12] SKIPPED - Already processed:
Naan

[13] SKIPPED - Already processed:
Chicken Chilli Chicken Kala Masala

[14] SKIPPED - Already processed:
Dal Tadka

[15] SKIPPED - Already processed:
Paneer Crispy

[16] SKIPPED - Already processed:
Butter Naan Chicken Crispy

[17] SKIPPED - Already processed:
Chicken Masala Boneless

[18] SKIPPED - Already processed:
Chicken Ta

IndexError: At least one sheet must be visible

In [42]:
excel_file = r"C:\Users\Bharg\Downloads\Assignment - Ai agent  - Sheet1.xlsx"

run_full_pipeline(
    excel_file,
    target_images=250
)

Already processed: 179
Target: 250
Remaining: 71

[1] SKIPPED:
Chicken Tandoori

[2] SKIPPED:
Veg Fried Rice

[3] SKIPPED:
Chicken Biryani Boneless

[4] SKIPPED:
Butter Roti

[5] SKIPPED:
Paneer Tikka Masala

[6] SKIPPED:
Chicken Fried Rice

[7] SKIPPED:
Veg Triple Schezwan Fried Rice

[8] SKIPPED:
Roti

[9] SKIPPED:
Chicken Tandoori

[10] SKIPPED:
Dal Khichdi Tadka Chicken Triple Fried Rice

[11] SKIPPED:
Chicken Lollipop

[12] SKIPPED:
Naan

[13] SKIPPED:
Chicken Chilli Chicken Kala Masala

[14] SKIPPED:
Dal Tadka

[15] SKIPPED:
Paneer Crispy

[16] SKIPPED:
Butter Naan Chicken Crispy

[17] SKIPPED:
Chicken Masala Boneless

[18] SKIPPED:
Chicken Tandoori

[19] SKIPPED:
Veg Handi

[20] SKIPPED:
Veg Dum Biryani

[21] SKIPPED:
Chicken Tikka Biryani

[22] SKIPPED:
Veg Manchurian

[23] SKIPPED:
Kulcha

[24] SKIPPED:
Veg Kolhapuri

[25] SKIPPED:
Chicken Schezwan Fried Rice

[26] SKIPPED:
Paneer Kolhapuri

[27] SKIPPED:
Paneer Makhanwala

[28] SKIPPED:
Chicken KL Special Soup

[29] SKIPPED:


C:\Users\Bharg\anaconda3\Lib\site-packages\PIL\Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Trying image 1...
[SKIPPED] Image 1: not a valid image
Trying image 2...
Blur score (higher is sharper): 483.13
[ACCEPTED] Image 2 passed quality check.
Original size: (1025, 1025)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Veg Schezwan Fried Rice Chicken Pahadi Kebab.jpg
Final size: 1800 × 1200 pixels
File size: 0.52 MB
SUCCESS: 208/250

[331] SKIPPED:
Dal Fry

[332] SKIPPED:
Chicken Hakka Noodles

PROCESSING
Food: Chicken KL Special Fried Rice Mushroom Chilli
Progress: 208/250

Searching image for: Chicken KL Special Fried Rice Mushroom Chilli...
Trying image 1...
Blur score (higher is sharper): 534.68
[ACCEPTED] Image 1 passed quality check.
Original size: (2000, 1333)
Processed size: (1800, 1200)
[PROCESSED & SAVED] processed_images\Chicken KL Special Fried Rice Mushroom Chilli.jpg
Final size: 1800 × 1200 pixels
File size: 0.39 MB
SUCCESS: 209/250

[334] SKIPPED:
Dal Khichdi

PROCESSING
Food: Chicken Lajawab
Progress: 209/250

Searching image for: Chicken Laj

IndexError: At least one sheet must be visible

In [40]:
import os

processed_folder = "processed_images"

files = [
    f for f in os.listdir(processed_folder)
    if f.lower().endswith(".jpg")
]

print("Processed images:", len(files))

Processed images: 179


In [43]:
import os
import cv2
import pandas as pd

def create_final_report(
    processed_dir="processed_images",
    report_dir="reports"
):

    os.makedirs(report_dir, exist_ok=True)

    results = []

    image_files = [
        f for f in os.listdir(processed_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(f"Images found: {len(image_files)}")

    for index, filename in enumerate(image_files, start=1):

        image_path = os.path.join(
            processed_dir,
            filename
        )

        img = cv2.imread(image_path)

        if img is None:
            results.append({
                "image_name": filename,
                "width": None,
                "height": None,
                "file_size_mb": None,
                "sharpness_score": None,
                "status": "INVALID_IMAGE"
            })
            continue

        height, width = img.shape[:2]

        gray = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        )

        sharpness = cv2.Laplacian(
            gray,
            cv2.CV_64F
        ).var()

        file_size_mb = (
            os.path.getsize(image_path)
            / (1024 * 1024)
        )

        if (
            width == 1800
            and height == 1200
            and file_size_mb <= 10
        ):
            status = "VALID"
        else:
            status = "CHECK_REQUIRED"

        results.append({
            "image_name": filename,
            "width": width,
            "height": height,
            "file_size_mb": round(file_size_mb, 2),
            "sharpness_score": round(sharpness, 2),
            "status": status
        })

    report_df = pd.DataFrame(results)

    report_path = os.path.join(
        report_dir,
        "final_image_quality_report.xlsx"
    )

    report_df.to_excel(
        report_path,
        index=False
    )

    valid_count = (
        report_df["status"] == "VALID"
    ).sum()

    check_count = (
        report_df["status"] == "CHECK_REQUIRED"
    ).sum()

    invalid_count = (
        report_df["status"] == "INVALID_IMAGE"
    ).sum()

    print("\n" + "=" * 50)
    print("FINAL IMAGE REPORT")
    print("=" * 50)

    print(f"Total images : {len(report_df)}")
    print(f"Valid        : {valid_count}")
    print(f"Check needed : {check_count}")
    print(f"Invalid      : {invalid_count}")

    print(f"\nReport saved to:")
    print(report_path)

    return report_df

In [44]:
final_report = create_final_report()

Images found: 250


IndexError: At least one sheet must be visible

In [1]:
import os
import cv2
import pandas as pd

processed_dir = "processed_images"
report_dir = "reports"

os.makedirs(report_dir, exist_ok=True)

results = []

image_files = [
    f for f in os.listdir(processed_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Images found:", len(image_files))

for filename in image_files:

    image_path = os.path.join(
        processed_dir,
        filename
    )

    img = cv2.imread(image_path)

    if img is None:
        results.append({
            "image_name": filename,
            "width": None,
            "height": None,
            "file_size_mb": None,
            "sharpness_score": None,
            "status": "INVALID_IMAGE"
        })
        continue

    height, width = img.shape[:2]

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    sharpness = cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

    file_size_mb = (
        os.path.getsize(image_path)
        / (1024 * 1024)
    )

    if (
        width == 1800
        and height == 1200
        and file_size_mb <= 10
    ):
        status = "VALID"
    else:
        status = "CHECK_REQUIRED"

    results.append({
        "image_name": filename,
        "width": width,
        "height": height,
        "file_size_mb": round(file_size_mb, 2),
        "sharpness_score": round(sharpness, 2),
        "status": status
    })

final_report = pd.DataFrame(results)

print("\nTotal:", len(final_report))
print("\nStatus:")
print(final_report["status"].value_counts())

C:\Users\Bharg\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Images found: 250

Total: 250

Status:
status
VALID    250
Name: count, dtype: int64


In [2]:
from openpyxl import Workbook
import os

report_path = os.path.join(
    "reports",
    "final_image_quality_report.xlsx"
)

wb = Workbook()
ws = wb.active
ws.title = "Image Quality Report"

headers = [
    "image_name",
    "width",
    "height",
    "file_size_mb",
    "sharpness_score",
    "status"
]

ws.append(headers)

# Add the data
for row in final_report.itertuples(index=False):
    ws.append(list(row))

# Save
wb.save(report_path)

print("Report successfully created!")
print(report_path)

Report successfully created!
reports\final_image_quality_report.xlsx


In [3]:
import os
import pandas as pd
from openpyxl import Workbook

# Load the original food list
food_list = load_and_clean_food_items(excel_file)

processed_dir = "processed_images"
report_dir = "reports"

os.makedirs(report_dir, exist_ok=True)

results = []

for food in food_list:

    original_name = food["original_name"]
    search_query = food["search_query"]

    clean_name = sanitize_filename(original_name)

    image_path = os.path.join(
        processed_dir,
        f"{clean_name}.jpg"
    )

    if not os.path.exists(image_path):
        continue

    matching_rows = final_report[
        final_report["image_name"] == f"{clean_name}.jpg"
    ]

    if len(matching_rows) > 0:

        row = matching_rows.iloc[0]

        results.append({
            "food_name": original_name,
            "search_query": search_query,
            "image_name": clean_name + ".jpg",
            "image_path": image_path,
            "width": row["width"],
            "height": row["height"],
            "file_size_mb": row["file_size_mb"],
            "sharpness_score": row["sharpness_score"],
            "status": row["status"]
        })


company_report = pd.DataFrame(results)

print("Food items matched:", len(company_report))

print("\nPreview:")
display(company_report.head(10))

NameError: name 'load_and_clean_food_items' is not defined

In [5]:
import pandas as pd
import re
import os
import cv2

excel_file = r"C:\Users\Bharg\Downloads\Assignment - Ai agent  - Sheet1.xlsx"
def load_and_clean_food_items(file_path):

    raw_df = pd.read_excel(
        file_path,
        header=None
    )

    header_row_idx = None

    for idx, row in raw_df.iterrows():

        row_values = (
            row.astype(str)
            .str.strip()
            .str.lower()
            .tolist()
        )

        if "item_name" in row_values:
            header_row_idx = idx
            break

    if header_row_idx is None:
        raise KeyError(
            "Could not locate 'item_name' in the Excel file."
        )

    df = pd.read_excel(
        file_path,
        header=header_row_idx
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    raw_items = (
        df["item_name"]
        .dropna()
        .tolist()
    )

    cleaned_items = []

    for item in raw_items:

        name = str(item).strip()

        if not name:
            continue

        if name.lower() == "item_name":
            continue

        name = re.sub(
            r"\[.*?\]",
            "",
            name
        )

        name = re.sub(
            r"\(.*?\)",
            "",
            name
        )

        individual_items = re.split(
            r"[\r\n]+",
            name
        )

        for individual_item in individual_items:

            individual_item = individual_item.strip()

            if not individual_item:
                continue

            cleaned_items.append({
                "original_name": individual_item,
                "search_query": individual_item
            })

    return cleaned_items

In [6]:
food_list = load_and_clean_food_items(excel_file)

print("Food items found:", len(food_list))
print(food_list[:5])

Food items found: 420
[{'original_name': 'Chicken Tandoori', 'search_query': 'Chicken Tandoori'}, {'original_name': 'Veg Fried Rice', 'search_query': 'Veg Fried Rice'}, {'original_name': 'Chicken Biryani Boneless', 'search_query': 'Chicken Biryani Boneless'}, {'original_name': 'Butter Roti', 'search_query': 'Butter Roti'}, {'original_name': 'Paneer Tikka Masala', 'search_query': 'Paneer Tikka Masala'}]


In [7]:
processed_dir = "processed_images"

results = []

image_files = [
    f for f in os.listdir(processed_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Images found:", len(image_files))

for filename in image_files:

    image_path = os.path.join(
        processed_dir,
        filename
    )

    img = cv2.imread(image_path)

    if img is None:
        continue

    height, width = img.shape[:2]

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    sharpness = cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

    file_size_mb = (
        os.path.getsize(image_path)
        / (1024 * 1024)
    )

    status = (
        "VALID"
        if width == 1800
        and height == 1200
        and file_size_mb <= 10
        else "CHECK_REQUIRED"
    )

    results.append({
        "image_name": filename,
        "width": width,
        "height": height,
        "file_size_mb": round(file_size_mb, 2),
        "sharpness_score": round(sharpness, 2),
        "status": status
    })

final_report = pd.DataFrame(results)

print("\nImages verified:", len(final_report))
print("\nStatus:")
print(final_report["status"].value_counts())

Images found: 250

Images verified: 250

Status:
status
VALID    250
Name: count, dtype: int64


In [8]:
company_report_data = []

for food in food_list:

    original_name = food["original_name"]
    search_query = food["search_query"]

    clean_name = re.sub(
        r'[\\/*?:"<>|]',
        "",
        original_name
    ).strip()

    image_filename = clean_name + ".jpg"

    image_path = os.path.join(
        "processed_images",
        image_filename
    )

    if not os.path.exists(image_path):
        continue

    matching_rows = final_report[
        final_report["image_name"] == image_filename
    ]

    if len(matching_rows) == 0:
        continue

    row = matching_rows.iloc[0]

    company_report_data.append({
        "food_name": original_name,
        "search_query": search_query,
        "image_name": image_filename,
        "image_path": image_path,
        "width": row["width"],
        "height": row["height"],
        "file_size_mb": row["file_size_mb"],
        "sharpness_score": row["sharpness_score"],
        "status": row["status"]
    })

company_report = pd.DataFrame(company_report_data)

print("Food-image matches:", len(company_report))

display(company_report.head(10))

Food-image matches: 412


,food_name,search_query,image_name,image_path,width,height,file_size_mb,sharpness_score,status
0,Chicken Tandoori,Chicken Tandoori,Chicken Tandoori.jpg,processed_images\Chicken Tandoori.jpg,1800,1200,0.39,402.05,VALID
1,Veg Fried Rice,Veg Fried Rice,Veg Fried Rice.jpg,processed_images\Veg Fried Rice.jpg,1800,1200,0.27,22.77,VALID
2,Chicken Biryani Boneless,Chicken Biryani Boneless,Chicken Biryani Boneless.jpg,processed_images\Chicken Biryani Boneless.jpg,1800,1200,0.58,186.78,VALID
3,Butter Roti,Butter Roti,Butter Roti.jpg,processed_images\Butter Roti.jpg,1800,1200,0.37,161.36,VALID
4,Paneer Tikka Masala,Paneer Tikka Masala,Paneer Tikka Masala.jpg,processed_images\Paneer Tikka Masala.jpg,1800,1200,0.42,183.26,VALID
5,Chicken Fried Rice,Chicken Fried Rice,Chicken Fried Rice.jpg,processed_images\Chicken Fried Rice.jpg,1800,1200,0.34,170.00,VALID
6,Veg Triple Schezwan Fried Rice,Veg Triple Schezwan Fried Rice,Veg Triple Schezwan Fried Rice.jpg,processed_images\Veg Triple Schezwan Fried Ric...,1800,1200,0.44,218.06,VALID
7,Roti,Roti,Roti.jpg,processed_images\Roti.jpg,1800,1200,0.37,161.36,VALID
8,Chicken Tandoori,Chicken Tandoori,Chicken Tandoori.jpg,processed_images\Chicken Tandoori.jpg,1800,1200,0.39,402.05,VALID
9,Dal Khichdi Tadka Chicken Triple Fried Rice,Dal Khichdi Tadka Chicken Triple Fried Rice,Dal Khichdi Tadka Chicken Triple Fried Rice.jpg,processed_images\Dal Khichdi Tadka Chicken Tri...,1800,1200,0.33,89.34,VALID


In [9]:
print("Total food records:", len(food_list))

print(
    "Unique food names:",
    len(set(food["original_name"] for food in food_list))
)

print(
    "Unique image files:",
    company_report["image_name"].nunique()
)

print(
    "Rows in company report:",
    len(company_report)
)

Total food records: 420
Unique food names: 258
Unique image files: 250
Rows in company report: 412


In [10]:
duplicates = company_report[
    company_report["food_name"].duplicated(keep=False)
].sort_values("food_name")

print("Duplicate food records:", len(duplicates))

display(duplicates.head(20))

Duplicate food records: 262


,food_name,search_query,image_name,image_path,width,height,file_size_mb,sharpness_score,status
73,Butter Garlic Naan,Butter Garlic Naan,Butter Garlic Naan.jpg,processed_images\Butter Garlic Naan.jpg,1800,1200,0.55,247.81,VALID
325,Butter Garlic Naan,Butter Garlic Naan,Butter Garlic Naan.jpg,processed_images\Butter Garlic Naan.jpg,1800,1200,0.55,247.81,VALID
85,Butter Naan,Butter Naan,Butter Naan.jpg,processed_images\Butter Naan.jpg,1800,1200,0.27,19.09,VALID
198,Butter Naan,Butter Naan,Butter Naan.jpg,processed_images\Butter Naan.jpg,1800,1200,0.27,19.09,VALID
295,Butter Naan,Butter Naan,Butter Naan.jpg,processed_images\Butter Naan.jpg,1800,1200,0.27,19.09,VALID
3,Butter Roti,Butter Roti,Butter Roti.jpg,processed_images\Butter Roti.jpg,1800,1200,0.37,161.36,VALID
76,Butter Roti,Butter Roti,Butter Roti.jpg,processed_images\Butter Roti.jpg,1800,1200,0.37,161.36,VALID
193,Butter Roti,Butter Roti,Butter Roti.jpg,processed_images\Butter Roti.jpg,1800,1200,0.37,161.36,VALID
288,Butter Roti,Butter Roti,Butter Roti.jpg,processed_images\Butter Roti.jpg,1800,1200,0.37,161.36,VALID
38,Chana Garlic Fry,Chana Garlic Fry,Chana Garlic Fry.jpg,processed_images\Chana Garlic Fry.jpg,1800,1200,0.32,17.14,VALID


In [11]:
missing_foods = []

for food in food_list:

    original_name = food["original_name"]

    clean_name = re.sub(
        r'[\\/*?:"<>|]',
        "",
        original_name
    ).strip()

    image_filename = clean_name + ".jpg"

    image_path = os.path.join(
        "processed_images",
        image_filename
    )

    if not os.path.exists(image_path):

        missing_foods.append({
            "food_name": original_name,
            "expected_image": image_filename
        })

missing_df = pd.DataFrame(missing_foods)

print("Food records without images:", len(missing_df))

display(missing_df)




Food records without images: 8


,food_name,expected_image
0,Mutton Pepper Soup,Mutton Pepper Soup.jpg
1,Chicken Hong Kong Fried Rice,Chicken Hong Kong Fried Rice.jpg
2,Chicken Manchurian Gravy,Chicken Manchurian Gravy.jpg
3,Chicken Jeera,Chicken Jeera.jpg
4,Paneer Kolhapuri Chicken Tandoor Masala,Paneer Kolhapuri Chicken Tandoor Masala.jpg
5,Aloo Matar,Aloo Matar.jpg
6,Paneer Pulao,Paneer Pulao.jpg
7,Chicken Cheese Garlic Balls,Chicken Cheese Garlic Balls.jpg


In [12]:

company_report_data = []

for food in food_list:

    original_name = food["original_name"]
    search_query = food["search_query"]

    clean_name = re.sub(
        r'[\\/*?:"<>|]',
        "",
        original_name
    ).strip()

    image_filename = clean_name + ".jpg"

    image_path = os.path.join(
        "processed_images",
        image_filename
    )

    # Image exists
    if os.path.exists(image_path):

        matching_rows = final_report[
            final_report["image_name"] == image_filename
        ]

        if len(matching_rows) > 0:

            row = matching_rows.iloc[0]

            company_report_data.append({
                "food_name": original_name,
                "search_query": search_query,
                "image_name": image_filename,
                "image_path": image_path,
                "width": row["width"],
                "height": row["height"],
                "file_size_mb": row["file_size_mb"],
                "sharpness_score": row["sharpness_score"],
                "status": row["status"]
            })

    else:

        company_report_data.append({
            "food_name": original_name,
            "search_query": search_query,
            "image_name": None,
            "image_path": None,
            "width": None,
            "height": None,
            "file_size_mb": None,
            "sharpness_score": None,
            "status": "IMAGE_NOT_AVAILABLE"
        })


company_report = pd.DataFrame(company_report_data)

print("Total food records:", len(company_report))

print("\nStatus:")
print(company_report["status"].value_counts())

display(company_report.head(10))

Total food records: 420

Status:
status
VALID                  412
IMAGE_NOT_AVAILABLE      8
Name: count, dtype: int64


,food_name,search_query,image_name,image_path,width,height,file_size_mb,sharpness_score,status
0,Chicken Tandoori,Chicken Tandoori,Chicken Tandoori.jpg,processed_images\Chicken Tandoori.jpg,1800.0,1200.0,0.39,402.05,VALID
1,Veg Fried Rice,Veg Fried Rice,Veg Fried Rice.jpg,processed_images\Veg Fried Rice.jpg,1800.0,1200.0,0.27,22.77,VALID
2,Chicken Biryani Boneless,Chicken Biryani Boneless,Chicken Biryani Boneless.jpg,processed_images\Chicken Biryani Boneless.jpg,1800.0,1200.0,0.58,186.78,VALID
3,Butter Roti,Butter Roti,Butter Roti.jpg,processed_images\Butter Roti.jpg,1800.0,1200.0,0.37,161.36,VALID
4,Paneer Tikka Masala,Paneer Tikka Masala,Paneer Tikka Masala.jpg,processed_images\Paneer Tikka Masala.jpg,1800.0,1200.0,0.42,183.26,VALID
5,Chicken Fried Rice,Chicken Fried Rice,Chicken Fried Rice.jpg,processed_images\Chicken Fried Rice.jpg,1800.0,1200.0,0.34,170.00,VALID
6,Veg Triple Schezwan Fried Rice,Veg Triple Schezwan Fried Rice,Veg Triple Schezwan Fried Rice.jpg,processed_images\Veg Triple Schezwan Fried Ric...,1800.0,1200.0,0.44,218.06,VALID
7,Roti,Roti,Roti.jpg,processed_images\Roti.jpg,1800.0,1200.0,0.37,161.36,VALID
8,Chicken Tandoori,Chicken Tandoori,Chicken Tandoori.jpg,processed_images\Chicken Tandoori.jpg,1800.0,1200.0,0.39,402.05,VALID
9,Dal Khichdi Tadka Chicken Triple Fried Rice,Dal Khichdi Tadka Chicken Triple Fried Rice,Dal Khichdi Tadka Chicken Triple Fried Rice.jpg,processed_images\Dal Khichdi Tadka Chicken Tri...,1800.0,1200.0,0.33,89.34,VALID


In [13]:
from openpyxl import Workbook
import os

os.makedirs("reports", exist_ok=True)

final_report_path = os.path.join(
    "reports",
    "company_image_processing_report.xlsx"
)

wb = Workbook()

ws = wb.active
ws.title = "Image Processing Report"

ws.append(list(company_report.columns))

# Add all 420 records
for row in company_report.itertuples(index=False):
    ws.append(list(row))

# Save the workbook
wb.save(final_report_path)

print("Final company report created!")
print(final_report_path)

Final company report created!
reports\company_image_processing_report.xlsx


In [14]:
print(
    "Report exists:",
    os.path.exists(final_report_path)
)

print(
    "Report location:",
    final_report_path
)

Report exists: True
Report location: reports\company_image_processing_report.xlsx
